In [ ]:
import numpy as np
import random
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchsurv.loss import cox

from sklearn.metrics import roc_auc_score, root_mean_squared_error, mean_squared_error
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.util import Surv

from ncps.torch import LTC
from ncps.wirings import AutoNCP

RAND_SEED = 5904
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load Data

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

NUM_FEATURES = train_X.shape[1]

train_surv_Y = Surv.from_arrays(train_Y[:,0], train_Y[:,1])
test_surv_Y = Surv.from_arrays(test_Y[:,0], test_Y[:,1])

## LTC LNN Model

In [ ]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.kaiming_normal_(module.weight, nonlinearity="leaky_relu")

        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

class ProbabilityPredictorLTC(nn.Module):
    def __init__(self, input_size, num_neurons, wiring : AutoNCP, batch_first=True, return_sequences=True, ode_unfolds=6):
        super(ProbabilityPredictorLTC, self).__init__()

        self.num_neurons = num_neurons

        self.ltc_lnn = LTC(input_size=input_size,
                            units=wiring,
                            batch_first=batch_first,
                            return_sequences=return_sequences,
                            ode_unfolds=ode_unfolds)

        self.dropout1 = nn.Dropout(0.5)
        self.dropout2 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(wiring.output_dim, int(wiring.output_dim * 2))
        self.normlayer1 = nn.LayerNorm(int(wiring.output_dim * 2))

        self.fc2 = nn.Linear(int(wiring.output_dim * 2), wiring.output_dim)
        self.normlayer2 = nn.LayerNorm(wiring.output_dim)

        self.fc3 = nn.Linear(wiring.output_dim, 1)

        self.leakyRelu = nn.LeakyReLU()
        self.apply(init_weights)

    def forward(self, input, timespans):

        x, _ = self.ltc_lnn(input=input, hx=None, timespans=timespans)

        # skip1 = x

        x = self.fc1(x)
        x = self.normlayer1(x)
        x = self.leakyRelu(x)
        x = self.dropout1(x)
        
        # x = x + skip1

        # skip2 = x

        x = self.fc2(x)
        x = self.normlayer2(x)
        x = self.leakyRelu(x)
        x = self.dropout2(x)

        # x = x + skip1

        output = self.fc3(x)

        # x = self.sigmoid(x)

        return output

def get_ltc_model(num_inputs, num_outputs, num_neurons, network_sparsity=0.5, ode_unfolds=6, return_sequences=True):
    
    network_wiring = AutoNCP(num_neurons, num_outputs, sparsity_level=network_sparsity, seed=RAND_SEED)

    model = ProbabilityPredictorLTC(input_size=num_inputs,
                                    num_neurons=num_neurons,
                                    wiring=network_wiring,
                                    batch_first=True,
                                    return_sequences=return_sequences,
                                    ode_unfolds=ode_unfolds)
    
    return model

In [ ]:
def get_data_sequences(features, event_times, labels, num_neurons, t_step):
    feature_sequences = list()
    time_sequences = list()
    label_sequences = list()
    sequence_masks = list()

    for feature_vector, time, label in zip(features, event_times, labels):

        divisions = time / t_step

        whole_divisions = int(divisions)
        remainder_divisions = divisions % 1

        num_time_steps = whole_divisions

        if whole_divisions > 0:
            time_seq = np.stack([t_step] * num_time_steps)
        else:
            time_seq = np.array([])

        if remainder_divisions > 0:
            time_seq = np.append(time_seq, (remainder_divisions * t_step))
            num_time_steps += 1

        assert time_seq.sum() == time

        time_seq = torch.tensor(time_seq)
        time_sequences.append(time_seq)

        # feature_seq = np.stack([feature_vector] * num_time_steps)
        # feature_seq = torch.tensor(feature_seq)
        # feature_sequences.append(feature_seq)

        label_seq = np.zeros_like(time_seq)
        label_seq[-1] = label
        label_seq = torch.tensor(label_seq)
        label_sequences.append(label_seq)

        seq_mask = np.ones_like(time_seq)
        seq_mask = torch.tensor(seq_mask)
        sequence_masks.append(seq_mask)

    times_T = pad_sequence(time_sequences, batch_first=True, padding_value=1e-8, padding_side="right")
    times_T = np.expand_dims(times_T, axis=-1)
    times_T = np.broadcast_to(times_T, (times_T.shape[0], times_T.shape[1], num_neurons))
    times_T = torch.tensor(times_T, dtype=torch.float32)

    features_X = np.expand_dims(features, axis=1)
    features_X = np.repeat(features_X, times_T.shape[1], axis=1)
    features_X = torch.tensor(features_X, dtype=torch.float32)

    # features_X = pad_sequence(feature_sequences, batch_first=True, padding_value=0, padding_side="right")
    # features_X = features_X.type(torch.float32)

    labels_Y = pad_sequence(label_sequences, batch_first=True, padding_value=0, padding_side="right")
    labels_Y = np.expand_dims(labels_Y, axis=-1)
    labels_Y = torch.tensor(labels_Y, dtype=torch.float32)

    masks_M = pad_sequence(sequence_masks, batch_first=True, padding_value=0, padding_side="right")
    masks_M = np.expand_dims(masks_M, axis=-1)
    masks_M = torch.tensor(masks_M, dtype=torch.float32)

    return features_X, times_T, labels_Y, masks_M

In [ ]:
class RelapseDataset(torch.utils.data.Dataset):
    def __init__(self, X, dt, Y, M):
        self.featuresX = X
        self.timesT = dt
        self.eventY = Y
        self.maskM = M

    def __len__(self):
        return len(self.featuresX)

    def __getitem__(self, idx):
        return self.featuresX[idx], self.timesT[idx], self.eventY[idx], self.maskM[idx]

def get_ltc_dataset(num_neurons, batch_size, time_step):
    # Stacking features along 2nd dimension to match the number of time steps

    train_features_X, train_times_T, train_labels_Y, train_masks_M = get_data_sequences(train_X, train_Y[:, 1], train_Y[:, 0], num_neurons, time_step)
    test_features_X, test_times_T, test_labels_Y, test_masks_M = get_data_sequences(test_X, test_Y[:, 1], test_Y[:, 0], num_neurons, time_step)

    # print(test_features_X.shape)
    # print(test_times_T.shape)
    # print(test_labels_Y.shape)

    training_data = RelapseDataset(train_features_X, train_times_T, train_labels_Y, train_masks_M)
    testing_data = RelapseDataset(test_features_X, test_times_T, test_labels_Y, test_masks_M)

    trainloader = torch.utils.data.DataLoader(training_data, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(testing_data, batch_size=batch_size, shuffle=False)

    seq_length = train_times_T.shape[1]

    return trainloader, testloader , seq_length

In [ ]:
trainloader, testloader, seq_length = get_ltc_dataset(64, 64, 0.5)

In [ ]:
def get_perf_metrics(model, dataloader, time_step, seq_length, device=torch.device("cpu")):
    predictions_y_hat = []
    # pred_sequences = []

    true_y = []
    time_vals = []

    # fixed_times_template = np.repeat(time_step, seq_length)

    # fixed_times_template[0] = 1.0
    # fixed_times_template[-1] -= 1.0

    # fixed_times_template = np.expand_dims(fixed_times_template, axis=(0,2))

    with torch.no_grad():
        model.eval()
        # model.ltc_lnn.return_sequences = True
        for data in dataloader:
            features, times, labels, masks = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            output = model(input=features, timespans=times)
            output = torch.sigmoid(output)
            output = output.squeeze(-1)

            # fixed_times = np.broadcast_to(fixed_times_template, (times.shape[0], seq_length, times.shape[2]))
            # fixed_times = torch.tensor(fixed_times, dtype=torch.float32)
            # fixed_times = fixed_times.to(device)

            # fixed_output = model(input=features, timespans=fixed_times)
            # fixed_output = torch.sigmoid(fixed_output)
            # fixed_output = fixed_output.squeeze(-1)

            # pred_sequences += fixed_output.cpu().tolist()

            batch_indexes = torch.arange(0, times.shape[0])
            event_indexes = masks.sum(dim=1) - 1
            event_indexes = event_indexes.reshape(-1).tolist()
            
            preds_at_event = output[batch_indexes, event_indexes].tolist()

            truths = labels.sum(dim=1).reshape(-1).tolist()
            event_times_t = times.sum(dim=1).T[0].tolist()

            # # Append to predictions list
            predictions_y_hat = predictions_y_hat + preds_at_event
            true_y = true_y + truths
            time_vals = time_vals + event_times_t
    
    # pred_surv = Surv.from_arrays(np.array(true_y).astype(bool), time_vals)

    # brier_times = np.arange(time_step, (times.shape[1] * time_step) + time_step, time_step)
    # brier_times[0] += 1.0
    # brier_times[-1] -= 1.0
    
    roc = roc_auc_score(true_y, predictions_y_hat)
    rmse = root_mean_squared_error(true_y, predictions_y_hat)
    # ibs = integrated_brier_score(train_surv_Y, pred_surv, pred_sequences, brier_times)
    concordance_data = concordance_index_censored(np.array(true_y).astype(bool), time_vals, predictions_y_hat)

    print("ROC: {:.5f}".format(roc), end=" ")
    print("RMSE: {:.5f}".format(rmse), end=" ")
    # print("IBS: {:.5f}".format(ibs))
    print("C-Index: {:.5f}".format(concordance_data[0]))

    model.train()

    return concordance_data[0]

## Training Model

In [ ]:
BATCH_SIZE = 64
BATCH_PRINT_STEP = 5
NUM_EPOCHS = 150
SEQ_TIME_STEP = 0.5

if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

num_neurons = 64
num_outputs = 12

model = get_ltc_model(NUM_FEATURES, num_outputs, num_neurons, network_sparsity=0.2, return_sequences=True)
trainloader, testloader, seq_length = get_ltc_dataset(num_neurons, BATCH_SIZE, time_step=SEQ_TIME_STEP)

model = model.to(device)

total_count = len(train_Y[:, 0])

pos_count = train_Y[:, 0].sum()
neg_count = total_count - pos_count

# print(total_count, pos_count, neg_count)
pos_weight = torch.tensor(neg_count / pos_count, device=device)

bce_criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
# criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)

best_score = 0.0

for epoch in range(0, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        features, times, labels, masks = data

        features = features.to(device)
        times = times.to(device)
        labels = labels.to(device)
        masks = masks.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward
        output = model(input=features, timespans=times)

        # BCE Loss
        bce_loss = bce_criterion(output, labels)
        masked_loss = masks * bce_loss
        bce_loss = masked_loss.sum() / masks.sum()

        # Cox Loss
        output = output.squeeze(-1)

        batch_indexes = torch.arange(0, times.shape[0])
        event_indexes = masks.sum(dim=1) - 1
        event_indexes = event_indexes.reshape(-1).tolist()
        
        preds_at_event = output[batch_indexes, event_indexes]

        truths = labels.sum(dim=1).reshape(-1).type(torch.bool)
        event_times_t = times.sum(dim=1).T[0]

        cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths, event_times_t)

        # Backward
        loss = cox_loss + (0.5 * bce_loss)

        loss.backward()
        
        # Optimize
        optimizer.step()

        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / BATCH_PRINT_STEP
            epochBatchLossPrint = "Epoch: {} Batch: {} Loss: {:.5f}".format(epoch + 1, i + 1, curr_loss)
            print(epochBatchLossPrint)
            running_loss = 0.0
    
    # Testing Loss
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data in testloader:
            features, times, labels, masks = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            output = model(input=features, timespans=times)
            
            # BCE Loss
            bce_loss = bce_criterion(output, labels)
            masked_loss = masks * bce_loss
            bce_loss = masked_loss.sum() / masks.sum()

            # Cox Loss
            output = output.squeeze(-1)

            batch_indexes = torch.arange(0, times.shape[0])
            event_indexes = masks.sum(dim=1) - 1
            event_indexes = event_indexes.reshape(-1).tolist()
            
            preds_at_event = output[batch_indexes, event_indexes]

            truths = labels.sum(dim=1).reshape(-1).type(torch.bool)
            event_times_t = times.sum(dim=1).T[0]

            cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths, event_times_t)

            # Joint Loss
            loss = cox_loss + (0.5 * bce_loss)

            val_loss += loss.item() * BATCH_SIZE
            
        val_loss /= len(testloader.dataset)

    print("Epoch: {} Testing Loss: {:.5f}".format(epoch + 1, val_loss))

    print("Training Data:")
    cindex_train = get_perf_metrics(model=model, dataloader=trainloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)
    print("Testing Data:")
    cindex_test = get_perf_metrics(model=model, dataloader=testloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)

    curr_score = cindex_train + cindex_test
    
    save_objective = cindex_test - (0.75 * abs(cindex_train - cindex_test))

    if(save_objective > best_score):
        best_score = save_objective
        print("New Model Weights Saved")
        torch.save(model.state_dict(), "weights_LTC.pth")

## Metrics

In [ ]:
if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model = model.to(device)

print("Training Data:")
get_perf_metrics(model=model, dataloader=trainloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)
print("Testing Data:")
get_perf_metrics(model=model, dataloader=testloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)

In [ ]:
state_dict = torch.load("../models/weights_LTC.pth", weights_only=True)
model.load_state_dict(state_dict)

In [ ]:
months = 62.5
num_steps = 120

time_diff = float(months) / float(num_steps)

# print(train_features_X[0].shape)
# print(train_times_T[0].shape)

# Convert a single test case to the right shape [Batch, Time, Neuron]
# In the ode solver, the time is used for calculations in each neuron.
# The calculations are performed elementwise so 
# Create list of time points

timespans_list = [time_diff] * num_steps

# Convert to numpy array
single_test_t = np.stack(timespans_list)

single_test_t = np.expand_dims(single_test_t, axis=-1)
single_test_t = np.broadcast_to(single_test_t, (single_test_t.shape[0], num_neurons))
single_test_t = torch.tensor(single_test_t).float().unsqueeze(0)
# print(single_test_t.shape)

# print(test_Y[:,0][test_index])

# Convert a single test case to the right shape [Batch, Vector, Feature]
test_index = 6
single_feature_vector = test_X[test_index]
print("label:", test_Y[:, 0][test_index])
print("time:",test_Y[:, 1][test_index])
# Make copies of the feature vector to match the number of prediction time points
copied_vectors = [single_feature_vector] * len(timespans_list)

single_test_X = np.stack(copied_vectors, axis=0)
# print(single_test_X.shape)
single_test_X = torch.tensor(single_test_X).float().unsqueeze(0)
# print(single_test_X)

with torch.no_grad():
    model.eval()
    model.to("cpu")

    model.ltc_lnn.return_sequences = True
    pred = model(input=single_test_X, timespans=single_test_t)
    pred = torch.sigmoid(pred)
    
pred_flatten = pred.reshape(-1).numpy()

# print(pred_flatten.shape)
# print(pred_flatten)

In [ ]:
x_values = np.arange(time_diff, months+time_diff, time_diff)

pred_rescaled = np.multiply(pred_flatten, time_diff)
cumulative_hazard = np.cumsum(pred_rescaled)

surv_func = np.exp(-cumulative_hazard)

plt.figure()
plt.step(x_values, surv_func, where="post")
plt.xlim([0, months])
plt.ylim([0, 1])
# plt.plot(x_values, surv_func)
plt.show()